# Storage Pantry — Instructor Key

Real yenstop snapshot from yen1, 2026-07-10 at 20:56. The script runs `top` in batch mode and captures **all processes** on the node — ~3000 rows, not ranked.

**Data path — change as needed:**
- Local Mac: `~/Downloads/yenstop_2026-07-10-20-56-06.csv`
- Yens scratch: `/scratch/shared/rf-bootcamp-2026/data/yenstop_2026-07-10-20-56-06.csv`

In [ ]:
import pandas as pd
import os

cols = ['timestamp','host','pid','user','pr','ni','virt','res','shr',
        's','cpu_pct','mem_pct','time_plus','command','type']

# Change this path as needed
DATA = os.path.expanduser('~/Downloads/yenstop_2026-07-10-20-56-06.csv')
# On the Yens:
# DATA = '/scratch/shared/rf-bootcamp-2026/data/yenstop_2026-07-10-20-56-06.csv'

df = pd.read_csv(DATA, header=None, names=cols)
print(f'{len(df)} rows, {df["user"].nunique()} unique users, {df["host"].nunique()} host(s)')
df.head(10)

## Q1 — Who is burning the most CPU?

**Expected:** `angikar` (python3) and `lenardst` (python) both at 96.9% — both running heavy compute jobs.  
Next: `topdump` at 46.9% — the monitoring script itself.  
**Discussion point:** two users pegging a full core each at the same time on a shared node.

In [ ]:
df.sort_values('cpu_pct', ascending=False)[['user','command','cpu_pct','s','time_plus']].head(10)

## Q2 — Running (R) vs sleeping (S)

**Expected:** vast majority are `S` (sleeping). Only a handful are `R` (actively using CPU).  
**Key insight:** sleeping processes still hold RAM — the fridge is occupied even when nobody is cooking.  
This is why shared nodes slow down: it's often RAM (fridge) filling up, not CPU (stove) being maxed.

In [ ]:
print(df['s'].value_counts())
print(f"\n{df[df['s']=='R']['user'].value_counts().to_string()}")

## Q3 — Who has been cooking the longest?

`time_plus` format is `HH:MM:SS` or `MM:SS.cc` — string sort works for the big ones since they lead with large numbers.

**Expected findings:**
- `muhua` — `remote-+` (likely code-server or remote desktop): **5030:18** ≈ 83 hours / 3.5 days
- `angikar` — `python3`: **1754:29** ≈ 29 hours
- `lenardst` — `python`: **1338:55** ≈ 22 hours

**Discussion point:** muhua's remote session has been alive for 3.5 days. This is common on the Yens — notebook kernels and remote sessions accumulate CPU time over days.

In [ ]:
# Sort by string length first (longer = more time), then alphabetically within same length
df['_tlen'] = df['time_plus'].str.len()
top_time = df.sort_values(['_tlen','time_plus'], ascending=False)[['user','command','time_plus','cpu_pct','s']].head(10)
df.drop(columns='_tlen', inplace=True)
top_time

## Q4 — VIRT vs RES puzzle (claude processes)

**Expected:** claude processes show enormous `virt` (tens of gigabytes) but small `res` (a few hundred MB).  

| user | virt | res |
|------|------|-----|
| merrigan | ~70 GB | ~440 MB |
| cxyou | ~77 GB | ~736 MB |

**Why:** Claude Code uses memory-mapped files extensively (the model weights, project context). The OS reserves virtual address space for these mappings but only loads pages into physical RAM on demand. So VIRT is huge, RES is what's actually in the fridge.  
**Key message for students:** always look at `res` (or `%mem`), not `virt`, when assessing memory pressure.

In [ ]:
claude = df[df['command'] == 'claude'][['user','virt','res','mem_pct','cpu_pct','s']]
print(claude.to_string(index=False))
print(f"\nVIRT/RES ratio for first claude process: {claude.iloc[0]['virt'] / claude.iloc[0]['res']:.0f}x")

## Bonus 1 — Total memory footprint per user

**Expected:** `muhua` likely at the top due to the large `remote-+` process. `lenardst` high due to 24GB resident Python job.  
**Discussion:** the user with the highest single-process CPU (`angikar`) may not be the biggest RAM consumer — two different bottlenecks.

In [ ]:
user_mem = df[df['type'] == 'u'].groupby('user')['mem_pct'].sum().sort_values(ascending=False)
print(user_mem.head(10).to_string())

## Bonus 2 — What are people running?

**Note:** `top` truncates command names to 8 characters and adds `+`.  
- `jupyter+` = `jupyterhub-singleuser` (notebook kernel)
- `remote-+` = `remote-viewer` or `code-server` or similar remote desktop tool
- `systemd+` = `systemd-journald` or similar system daemon
- `falcon-+` = `falcon-sensor` (CrowdStrike endpoint security agent)

**Discussion point:** students can look up PIDs they recognize from earlier exercises, or use `ps -p PID -o comm=` on the Yens to get full names.

In [ ]:
print('=== User processes ===')
print(df[df['type'] == 'u']['command'].value_counts().to_string())
print('\n=== System processes ===')
print(df[df['type'] == 's']['command'].value_counts().to_string())

## Full picture — who is on this node?

Useful for the class discussion wrap-up.

In [ ]:
summary = df[df['type'] == 'u'].groupby('user').agg(
    processes=('pid', 'count'),
    total_cpu=('cpu_pct', 'sum'),
    total_mem=('mem_pct', 'sum'),
    running=('s', lambda x: (x == 'R').sum()),
    tools=('command', lambda x: ', '.join(sorted(x.unique())))
).sort_values('total_cpu', ascending=False)

summary